# 股價創高分析

分析股票每次創歷史新高（ATH）及 Rolling Window 新高後：
1. 後續要多久才達到下一個新高點
2. 創高之後多久開始下跌

In [ ]:
# === 設定區 ===
TICKER = "SPY"           # 股票代碼（如 2330.TW、AAPL、SPY）
START_DATE = "2000-01-01" # 起始日期
END_DATE = None           # None = 到今天
INTERVAL = "1d"          # 時間粒度：1d / 1wk / 1mo
ROLLING_WINDOW = 252     # Rolling 新高的回看天數（252 ≈ 52 週）

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas as pd
import yfinance as yf

sys.path.insert(0, str(Path.cwd().parent))
from src.peak_detector import find_ath_peaks, find_rolling_peaks

In [ ]:
# 資料抓取與快取
data_dir = Path.cwd().parent / "data"
data_dir.mkdir(exist_ok=True)
cache_file = data_dir / f"{TICKER.replace('.', '_')}_{INTERVAL}.csv"

if cache_file.exists():
    df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
    print(f"從快取載入: {cache_file}")
else:
    df = yf.download(TICKER, start=START_DATE, end=END_DATE, interval=INTERVAL)
    # yfinance 回傳 MultiIndex columns，需要 flatten
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.to_csv(cache_file)
    print(f"下載完成，已快取至: {cache_file}")

print(f"資料範圍: {df.index[0].date()} ~ {df.index[-1].date()}，共 {len(df)} 筆")

## ATH（歷史新高）分析

In [ ]:
df_ath = find_ath_peaks(df)
ath_points = df_ath[df_ath["is_ath"]]
print(f"ATH 山峰數: {len(ath_points)}")

# 統計摘要
stats_next = ath_points["days_to_next_ath"].dropna()
stats_decline = ath_points["days_to_decline"].dropna()
stats_drawdown = ath_points["max_drawdown_pct"].dropna()

summary = pd.DataFrame({
    "到下一個 ATH 天數": [stats_next.mean(), stats_next.median(), stats_next.min(), stats_next.max()],
    "創高後開始下跌天數": [stats_decline.mean(), stats_decline.median(), stats_decline.min(), stats_decline.max()],
    "最大回撤 %": [stats_drawdown.mean(), stats_drawdown.median(), stats_drawdown.min(), stats_drawdown.max()],
}, index=["平均", "中位數", "最小", "最大"])
summary

In [ ]:
# K 線圖 + ATH 標記
ath_markers = pd.Series(float("nan"), index=df_ath.index)
ath_markers[df_ath["is_ath"]] = df_ath.loc[df_ath["is_ath"], "High"] * 1.02

ap = mpf.make_addplot(ath_markers, type="scatter", markersize=30, marker="v", color="red")
mpf.plot(
    df_ath.tail(500),
    type="candle",
    style="charles",
    title=f"{TICKER} K線圖 + ATH 標記（近 500 交易日）",
    addplot=mpf.make_addplot(ath_markers.tail(500), type="scatter", markersize=30, marker="v", color="red"),
    figsize=(16, 6),
)

In [ ]:
# 分布圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(stats_next, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(stats_next.median(), color="red", linestyle="--", label=f"中位數: {stats_next.median():.0f} 天")
axes[0].set_title("兩次 ATH 間隔天數分布")
axes[0].set_xlabel("交易日數")
axes[0].legend()

axes[1].hist(stats_decline, bins=30, color="darkorange", edgecolor="white")
axes[1].axvline(stats_decline.median(), color="red", linestyle="--", label=f"中位數: {stats_decline.median():.0f} 天")
axes[1].set_title("ATH 後開始下跌天數分布")
axes[1].set_xlabel("交易日數")
axes[1].legend()

plt.tight_layout()
plt.show()

## Rolling Window 新高分析

In [ ]:
df_rolling = find_rolling_peaks(df, window=ROLLING_WINDOW)
rolling_points = df_rolling[df_rolling["is_rolling_peak"]]
print(f"{ROLLING_WINDOW} 日新高山峰數: {len(rolling_points)}")

stats_next_r = rolling_points["days_to_next_rolling_peak"].dropna()
stats_decline_r = rolling_points["days_to_decline_from_rolling"].dropna()
stats_drawdown_r = rolling_points["max_drawdown_pct_rolling"].dropna()

summary_r = pd.DataFrame({
    "到下一個新高天數": [stats_next_r.mean(), stats_next_r.median(), stats_next_r.min(), stats_next_r.max()],
    "創高後開始下跌天數": [stats_decline_r.mean(), stats_decline_r.median(), stats_decline_r.min(), stats_decline_r.max()],
    "最大回撤 %": [stats_drawdown_r.mean(), stats_drawdown_r.median(), stats_drawdown_r.min(), stats_drawdown_r.max()],
}, index=["平均", "中位數", "最小", "最大"])
summary_r

In [ ]:
# 分布圖 - Rolling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(stats_next_r, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(stats_next_r.median(), color="red", linestyle="--", label=f"中位數: {stats_next_r.median():.0f} 天")
axes[0].set_title(f"兩次 {ROLLING_WINDOW}日新高 間隔天數分布")
axes[0].set_xlabel("交易日數")
axes[0].legend()

axes[1].hist(stats_decline_r, bins=30, color="darkorange", edgecolor="white")
axes[1].axvline(stats_decline_r.median(), color="red", linestyle="--", label=f"中位數: {stats_decline_r.median():.0f} 天")
axes[1].set_title(f"{ROLLING_WINDOW}日新高後 開始下跌天數分布")
axes[1].set_xlabel("交易日數")
axes[1].legend()

plt.tight_layout()
plt.show()

## 結論

以上分析顯示了股票創高後的行為模式：
- **兩次創高間隔**：中位數越大，代表突破新高越困難
- **創高後下跌速度**：中位數越小，代表創高後很快就會回落（追高風險高）
- 可調整 `TICKER` 和 `ROLLING_WINDOW` 觀察不同標的和不同定義下的表現